In [1]:
#q = 131
q = 179
m = 128
r = 138
n = 149
k = 56

## Key Generation

In [2]:
F.<z> = GF(q)
Flist = F.list()
Fstar = [i for i in F if i != 0]

def random_permutation_matrix(n):
    perm = Permutations(n).random_element()
    M = matrix(F, n, n)
    for i in range(n):
        M[i, perm(i+1)-1] = F(1)
    return M    

def homogeneous(n, m):
    M = zero_matrix(F, n, m)
    M[:m, :m] = random_permutation_matrix(m)
    M[m:, :n-m] = identity_matrix(F, n-m)
    return random_permutation_matrix(n) * M * random_permutation_matrix(m) 

def part_perm(r, n):
    M = zero_matrix(F, r, n)
    M[:, :r] = identity_matrix(F, r)
    return M * random_permutation_matrix(n)

def gen_error(n, t):
    res = zero_vector(F, n)
    for i in sample(range(n), t):
        res[i] = choice(Fstar)
    return res

In [3]:
alpha = sample(F.list(), m)
beta = [choice(Fstar) for i in range(m)]
C = codes.GeneralizedReedSolomonCode(alpha, k, beta)
S = random_matrix(F, k)
while S.rank() != k:
    S = random_matrix(F, k)
G = S*C.generator_matrix()

G1 = random_matrix(F, k, n)
while G1.rank() != k:
    G1 = random_matrix(F, k, n)
    
M1 = random_matrix(F, n)
M2 = random_matrix(F, n)
while M1.rank() != n:
    M1 = random_matrix(F, n)
while M2.rank() != n:
    M2 = random_matrix(F, n)

Q = part_perm(r, n)
R = homogeneous(n, m)
sol1 = (G1*M1*M2^-1).solve_right(G).T
sol2 = (G1*M1*M2^-1).right_kernel_matrix()
H2 = sol1 + random_matrix(F, sol1.nrows(), sol2.nrows()) * sol2
while H2.rank() != m: 
    sol2 = (G1*M^-1).right_kernel_matrix()
    H2 = sol1 + random_matrix(F, sol1.nrows(), sol2.nrows()) * sol2
P = (H2.T).solve_left(R)
G2 = (H2.T).left_kernel_matrix()
U = random_matrix(F, P.nrows(), G2.nrows())

Gpub = G1*M1
Epub = Q*(P+U*G2)*M2

In [4]:
# checking colunm weights in QR
QR = Q*R
c0 = 0
c1 = 0
c2 = 0
for i in (QR).columns():
    if i.hamming_weight() == 0:
        c0 += 1
    if i.hamming_weight() == 1:
        c1 += 1
    if i.hamming_weight() == 2:
        c2 += 1 
print(c0, c1, c2)

10 98 20


In [5]:
# Testing
msg = random_vector(F, k)
err = gen_error(r, (m-k) >> 1)
ciphertext = msg*Gpub + err * Epub
C.decode_to_message(ciphertext*M2^-1*(H2.T)) == msg * S

True

In [6]:
err.hamming_weight()

36

## Step 0: Finding auxilary matrices

In [7]:
K = Epub.right_kernel_matrix().T
K.rank(), K.rank() == n - r

(11, True)

In [8]:
K_ = Gpub*K
zeta = K_.rank()
A = K_.augment(identity_matrix(F, k)).rref()[:, K_.ncols():]
A*K_ == K_.rref()

True

In [9]:
U = A[zeta:] * Gpub

## Step 1.1: recovering the set of weight-1 columns of QR

An auxilary distingusher of non-GRS columns:

In [10]:
def square(G):
    N = G.nrows()
    res = zero_matrix(F, N*(N-1)//2 + N, G.ncols())
    r = 0
    for i in range(N):
        for j in range(i, N):
            res[r] = G[i].pairwise_product(G[j])
            r += 1
    return res

def remove_rand_columns_pass(G):
    if square(G).rank() == G.ncols():
        ns = 1
        B = codes.LinearCode(G).shortened([i for i in range(G.ncols() - ns, G.ncols())]).generator_matrix()
        while square(B).rank() >= G.ncols() - 3:
            ns += 1
            B = codes.LinearCode(G).shortened([i for i in range(G.ncols() - ns, G.ncols())]).generator_matrix()
        rr = square(B).rank()
        rnd_ind = []
        for i in range(0, G.ncols() - ns - 1):
            D = B.delete_columns([i])
            if square(D).rank() == rr - 1:
                rnd_ind.append(i)
        AA = G.delete_columns(rnd_ind)
        ns = 0
        B = AA#codes.LinearCode(AA).shortened([i for i in range(ns)]).generator_matrix()
        while square(B).rank() >= B.ncols() - 3:
            ns += 1
            B = codes.LinearCode(AA).shortened([i for i in range(ns)]).generator_matrix()
        rr = square(B).rank()
        rnd_ind = []
        for i in range(ns, AA.ncols()):
            D = B.delete_columns([i-ns])
            if square(D).rank() == rr - 1:
                rnd_ind.append(i)
        return AA.delete_columns(rnd_ind)
    else:
        rr = square(G).rank()
        rnd_ind = []
        for i in range(G.ncols()):
            if square(G.delete_columns([i])).rank() == rr-1:
                rnd_ind.append(i)
        return G.delete_columns(rnd_ind)

def remove_rand_columns(G):
    AA = remove_rand_columns_pass(G)
    return remove_rand_columns_pass(AA)

def get_punctured_indices(original_matrix, punctured_matrix):
    original_cols = original_matrix.columns()
    punctured_cols = punctured_matrix.columns()
    punctured_indices = []
    for i, col in enumerate(original_cols):
        if col not in punctured_cols:
            punctured_indices.append(i)    
    return punctured_indices

In [11]:
# Finding L
L = Epub.solve_right(identity_matrix(F, r))

# getting rid of $E_{pub}$'s kernel 
T = U*L

# Getting rid of non-GRS columns in T
T_J = remove_rand_columns(T)

T.ncols(), T_J.ncols()

(138, 98)

In [12]:
J = get_punctured_indices(T, T_J)

W_J = block_matrix([
    [identity_matrix(F, r)[:, i] for i in range(r) if i not in J]
])

# Let's check number of common columns of QR_ and ground truth QR
def common_columns(A, B):
    common_count = 0
    for col_A in A.columns():
        for col_B in B.columns():
            if col_A == col_B:
                common_count += 1
                break
    return common_count

# checking
W_J.ncols(), c1, common_columns(W_J, QR)

(98, 98, 98)

## Step 1.2: recovering weight-2 columns of QR

In [13]:
def zero_rows(G):
    zero_indices = [i for i in range(G.nrows()) if G.row(i).is_zero()]
    return zero_indices

# def checkGRS(G):
#     G2 = remove_rand_columns_pass(G)
#     if G == G2:
#         return True
#     return False

# slightly more efficient version of the previous function
def checkGRS(G):
    if square(G).rank() == G.ncols():
        ns = 0
        G1 = codes.LinearCode(G).shortened([i for i in range(ns)]).generator_matrix()
        while square(G1).rank() >= G1.ncols() - 2:
            ns += 1
            G1 = codes.LinearCode(G).shortened([i for i in range(ns)]).generator_matrix()
        rr = square(G1).rank()
        D = G1[:, :-1]
        if square(D).rank() == rr - 1:
            return False
    else:
        rr = square(G).rank()
        if square(G[:, :-1]).rank() == rr-1:
            return False
    return True

def find_new_column(B):
    zero_row_inds = zero_rows(B)
    for i in range(len(zero_row_inds)):
        for j in range(i+1, len(zero_row_inds)):
            new_col = zero_vector(F, r)
            new_col[zero_row_inds[i]] = 1
            new_col[zero_row_inds[j]] = 1
            B_test = B.augment(new_col)
            T =  U * Epub.solve_right(B_test)
            if checkGRS(T):
                return B_test
    return B

In [14]:
B = copy(W_J)
while B.ncols() != m:
    old_B = copy(B)
    B = find_new_column(B)
    if B == old_B:
        break
    print("+")

+
+
+
+
+
+
+
+
+
+
+
+
+
+
+
+
+
+
+
+


In [15]:
# checking
B.ncols(), c1 + c2, common_columns(B, QR)

(118, 118, 118)

## Step 2.1: recovering the structure of $\widetilde{C}$ 

In [16]:
Y = Epub.solve_right(B)

G3 = U * Y

In [17]:
def recover_GRS_support(G, x0, x1, xk):
    n = G.ncols()
    k = G.nrows()
    G1 = G.rref()
    x = [q+1]*n
    x[0], x[1], x[k] = x0, x1, xk
    count = 3
    i = 0; i_ = 1; j_ = k
    for j in range(k+1, n):
        gamma = (G1[i,j] * G1[i_, j_]) / (G1[i, j_] * G1[i_, j])
        denom = (x[j_] - x[i]) - gamma*(x[j_] - x[i_])
        if denom != 0:
            xj = (x[i_]*(x[j_]-x[i]) - gamma*x[i]*(x[j_] - x[i_])) / denom
            if xj in x:
                return x, False
            x[j] = xj
        else:
            return x, False
    i_ = 0; j = k; j_ = k+1
    for i in range(2, k):
        gamma = (G1[i,j] * G1[i_, j_]) / (G1[i, j_] * G1[i_, j])
        denom = gamma*(x[j_] - x[i_]) - (x[j] - x[i_])
        if denom != 0:
            xi = (gamma*x[j]*(x[j_]-x[i_]) - (x[j] - x[i_])*x[j_]) / denom
            if xi in x:
                return x, False
            x[i] = xi
        else:
            return x, False            
    return x, True

In [18]:
candidate_supports = []

G3_sq = codes.LinearCode(square(G3)).generator_matrix().rref()
x0, x1 = 0, 1
for xk in F.list():
    if (xk == 0) or (xk == 1):
        continue
    fl = False
    new_x, fl = recover_GRS_support(G3_sq, x0, x1, xk)
    if fl:
        candidate_supports.append(new_x)
     
print(len(candidate_supports))
#print(candidate_supports[0])

62


In [19]:
X_new = candidate_supports[0]

num_of_unknowns = G3.ncols()
I_nou = identity_matrix(F, num_of_unknowns)
RS_pc_mat = codes.GeneralizedReedSolomonCode(X_new, k - c0).parity_check_matrix()
tmp = matrix([(G3 * diagonal_matrix(i.list()) * RS_pc_mat.T).list() for i in I_nou])
sol = tmp.left_kernel().random_element()
Y_new = [i^-1 for i in sol]

In [20]:
C_rec = codes.GeneralizedReedSolomonCode(X_new, k - c0, Y_new)

## Step 2.2: decoding

In [21]:
msg_part = ciphertext*K
msg_part

(132, 57, 37, 65, 46, 154, 44, 88, 176, 63, 21)

In [22]:
noisy_codeword = ciphertext*Y - msg_part * A[0:len(msg_part)] * Gpub * Y
noisy_codeword

(128, 44, 91, 120, 149, 108, 151, 133, 71, 162, 64, 4, 19, 134, 68, 177, 41, 130, 109, 137, 159, 147, 150, 59, 10, 27, 40, 167, 118, 75, 39, 86, 18, 25, 165, 5, 98, 11, 108, 61, 41, 54, 135, 149, 75, 176, 174, 31, 12, 131, 96, 19, 46, 149, 79, 137, 57, 150, 153, 2, 30, 4, 178, 142, 136, 142, 127, 67, 164, 109, 136, 156, 16, 176, 34, 163, 7, 108, 36, 16, 75, 34, 97, 62, 93, 90, 0, 78, 156, 7, 10, 39, 118, 14, 87, 141, 23, 112, 70, 17, 90, 57, 158, 99, 137, 136, 173, 127, 98, 22, 135, 5, 125, 133, 99, 39, 129, 155)

In [23]:
codeword = C_rec.decode_to_code(noisy_codeword)
codeword

(128, 44, 83, 120, 149, 108, 22, 133, 71, 162, 142, 4, 19, 134, 68, 177, 41, 130, 109, 137, 159, 147, 108, 59, 74, 38, 40, 167, 118, 75, 149, 86, 18, 25, 61, 5, 98, 11, 114, 61, 50, 54, 171, 149, 75, 176, 174, 31, 12, 131, 96, 19, 46, 149, 79, 137, 57, 150, 153, 2, 30, 118, 77, 8, 136, 142, 97, 67, 164, 119, 136, 175, 167, 176, 34, 163, 7, 108, 8, 16, 75, 34, 50, 62, 93, 90, 0, 78, 156, 20, 33, 39, 121, 14, 143, 90, 112, 112, 159, 17, 90, 163, 62, 122, 137, 136, 11, 89, 98, 22, 89, 5, 125, 125, 99, 39, 129, 171)

In [24]:
recovered_plaintext = vector(list(msg_part) + list(G3.solve_left(codeword))) * A
recovered_plaintext

(117, 107, 113, 102, 108, 80, 173, 138, 4, 21, 52, 153, 173, 1, 97, 175, 77, 64, 60, 38, 112, 51, 165, 150, 138, 55, 42, 173, 101, 136, 42, 88, 75, 73, 162, 34, 114, 15, 72, 97, 114, 110, 172, 48, 87, 109, 164, 126, 58, 134, 88, 146, 39, 9, 159, 108)

In [25]:
recovered_plaintext == msg

True